# BanglaGuard XLM-R Base — Cross-Domain (LODO) Analysis — Colab

This is the Colab version of the uploaded BanglaGuard XLM-R Base LODO notebook.

It runs only:

- Healthcare + E-commerce → unseen Banking
- Banking + E-commerce → unseen Healthcare
- Banking + Healthcare → unseen E-commerce

For each fold, the target domain is excluded from training, validation,
TAPT/TLM, and rationale alignment. Target-domain inference uses the generic
policy embedding.

## How to run

1. Colab: **Runtime → Change runtime type → GPU**.
2. Run Cell 1.
3. Cell 2 asks you to upload the dataset from your computer.
4. Upload **`splits_v2.zip`** (recommended), or directly upload
   **`SecBangla_clean_master_with_splits.csv`**.
5. First keep `SMOKE_RUN=True`.
6. When all 3 smoke folds pass, change it to `False`.
7. Full mode runs seeds `42, 43, 44`.
8. At the end, Colab downloads the result ZIP automatically.

In [ ]:
# ============================================================
# CELL 1 — INSTALL + COLAB CONFIG
# ============================================================
!pip -q install -U "transformers>=4.50,<5" "accelerate>=1.0" \
    "sentencepiece>=0.1.99" "safetensors>=0.4" "scikit-learn>=1.3"

import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import re, gc, json, math, time, random, shutil, zipfile, warnings, unicodedata
from pathlib import Path
from contextlib import nullcontext

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score,
    precision_recall_fscore_support, matthews_corrcoef
)

from transformers import (
    AutoTokenizer, AutoModel, AutoModelForMaskedLM,
    DataCollatorForLanguageModeling, get_linear_schedule_with_warmup
)

from google.colab import files as colab_files

warnings.filterwarnings("ignore")

BACKBONE = "FacebookAI/xlm-roberta-base"
MODEL_SLUG = "banglaguard_xlmr_base_lodo_colab"

# ---------------- Run controls ----------------
SMOKE_RUN = True

RUN_ALL_SEEDS = True
SINGLE_SEED = 42
FULL_SEEDS = [42, 43, 44]
FIXED_HEAD_INIT_SEED = 2026

LODO_TARGETS = ["banking", "healthcare", "e-commerce"]

USE_TAPT_TLM = True
USE_RATIONALE_ALIGNMENT = True

TEXT_VIEWS = ["bangla", "banglish", "code_mix", "english"]

LABELS = ["ALLOW", "REVIEW", "BLOCK"]
LABEL2ID = {x:i for i,x in enumerate(LABELS)}
ID2LABEL = {i:x for x,i in LABEL2ID.items()}

DOMAINS = ["banking", "healthcare", "e-commerce"]
DOMAIN2ID = {x:i for i,x in enumerate(DOMAINS)}

INTENTS = [
    "B0_legitimate_security",
    "S1_jailbreak",
    "S2_prompt_extraction",
    "S3_data_exfiltration",
    "S4_unauthorized_action",
    "S5_authentication_authorization_bypass",
    "S6_indirect_injection",
]
INTENT2ID = {x:i for i,x in enumerate(INTENTS)}

RISKS = ["none", "low", "medium", "high", "very_high"]
RISK2ID = {x:i for i,x in enumerate(RISKS)}

# ---------------- Training ----------------
MAX_LEN = 128
EPOCHS = 12
PATIENCE = 3
EARLY_STOP_MIN_DELTA = 1e-3

LR_ENCODER = 3e-5
LR_HEADS = 3e-4
LAYERWISE_LR_DECAY = 0.90
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10
MAX_GRAD_NORM = 1.0

DROPOUT = 0.10
POLICY_DROPOUT = 0.20
LABEL_SMOOTHING = 0.05
AUGMENT_PROB = 0.30

LAMBDA_INT = 0.50
LAMBDA_DOM = 0.10
LAMBDA_ORD = 0.30
LAMBDA_CONS = 1.00
LAMBDA_SCL = 0.10
LAMBDA_RAT = 0.20
SUPCON_TAU = 0.07

CLASS_W = torch.tensor([0.96, 1.30, 0.74], dtype=torch.float32)

# Conservative Colab/T4 defaults.
# Effective scenario batch is still 16.
TRAIN_SCENARIO_BATCH = 1
GRAD_ACCUM = 16
EVAL_SCENARIO_BATCH = 2

TAPT_EPOCHS = 3
TAPT_LR = 3e-5
TAPT_BATCH = 2
TAPT_GRAD_ACCUM = 4

RATIONALE_MODEL = "intfloat/multilingual-e5-large"

SMOKE_TRAIN_N = 48
SMOKE_VAL_N = 24
SMOKE_TEST_N = 36
SMOKE_MATCHED_N = 24
SMOKE_MAX_STEPS = 6
SMOKE_TAPT_STEPS = 4

N_BOOT = 100 if SMOKE_RUN else 2000

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
CC = torch.cuda.get_device_capability(0) if torch.cuda.is_available() else (0,0)

AMP_DTYPE = (
    torch.bfloat16
    if (torch.cuda.is_available() and CC[0] >= 8)
    else torch.float16
)
USE_SCALER = torch.cuda.is_available() and AMP_DTYPE == torch.float16

WORK_ROOT = Path("/content")
UPLOAD_ROOT = WORK_ROOT / "runtime_uploads"
UPLOAD_ROOT.mkdir(parents=True, exist_ok=True)

RUN_MODE = "smoke" if SMOKE_RUN else "full"
OUT_DIR = WORK_ROOT / MODEL_SLUG / RUN_MODE
OUT_DIR.mkdir(parents=True, exist_ok=True)

def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def free_memory():
    gc.collect()
    if torch.cuda.is_available():
        try:
            torch.cuda.synchronize()
        except Exception:
            pass
        torch.cuda.empty_cache()
        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass
        gc.collect()

def amp_context():
    if torch.cuda.is_available():
        return torch.autocast("cuda", dtype=AMP_DTYPE)
    return nullcontext()

ACTIVE_SEEDS = (
    [42]
    if SMOKE_RUN
    else (FULL_SEEDS if RUN_ALL_SEEDS else [SINGLE_SEED])
)

print("GPU          :", GPU_NAME)
print("AMP dtype    :", AMP_DTYPE)
print("run mode     :", RUN_MODE)
print("LODO targets :", LODO_TARGETS)
print("seeds        :", ACTIVE_SEEDS)
print("output       :", OUT_DIR)

if not torch.cuda.is_available():
    print("\nWARNING: GPU is disabled. Use Runtime → Change runtime type → GPU.")

In [ ]:
# ============================================================
# CELL 2 — UPLOAD DATASET INTO COLAB RUNTIME
# ============================================================
print("Upload:")
print("  • splits_v2.zip  (recommended)")
print("or")
print("  • SecBangla_clean_master_with_splits.csv")
print()

uploaded = colab_files.upload()

if not uploaded:
    raise RuntimeError("No file uploaded. Re-run this cell and select the dataset.")

for name, payload in uploaded.items():
    dst = UPLOAD_ROOT / Path(name).name
    with open(dst, "wb") as f:
        f.write(payload)
    print(f"Saved {dst}  ({len(payload)/1024**2:.2f} MB)")

print("Upload complete.")

In [ ]:
# ============================================================
# CELL 3 — RESOLVE RUNTIME UPLOAD + BUILD STRICT LODO SPLITS
# ============================================================
MASTER_FILENAME = "SecBangla_clean_master_with_splits.csv"
EXTRACT_ROOT = WORK_ROOT / "splits_v2_runtime"
EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)

def find_master(root):
    hits = list(Path(root).rglob(MASTER_FILENAME))
    return hits[0] if hits else None

master_path = find_master(UPLOAD_ROOT)

if master_path is None:
    zip_files = list(UPLOAD_ROOT.glob("*.zip"))
    if not zip_files:
        raise FileNotFoundError(
            f"Neither {MASTER_FILENAME} nor a ZIP file was found. "
            "Re-run Cell 2 and upload splits_v2.zip."
        )

    for z in zip_files:
        print("Extracting:", z.name)
        try:
            with zipfile.ZipFile(z, "r") as zf:
                zf.extractall(EXTRACT_ROOT)
        except zipfile.BadZipFile as e:
            raise RuntimeError(f"{z.name} is not a valid ZIP archive.") from e

        master_path = find_master(EXTRACT_ROOT)
        if master_path is not None:
            break

if master_path is None:
    raise FileNotFoundError(
        f"Extraction completed, but {MASTER_FILENAME} was not found."
    )

DATA_DIR = master_path.parent
MASTER = pd.read_csv(master_path)

print("Master CSV:", master_path)
print("Rows      :", len(MASTER))

REQUIRED = [
    "id", "domain", "security_intent", "hard_negative_type",
    "decision", "risk_level", "note",
    "bangla", "banglish", "code_mix", "english",
    "scenario_family_id", "split", "in_decision_matched_test",
]

missing = [c for c in REQUIRED if c not in MASTER.columns]
assert not missing, f"Missing required columns: {missing}"
assert MASTER.id.is_unique
assert set(MASTER.domain) == set(DOMAINS)
assert set(MASTER.decision) <= set(LABELS)

for v in TEXT_VIEWS:
    assert MASTER[v].notna().all(), f"Null text in {v}"

def bool_series(s):
    if s.dtype == bool:
        return s
    return s.astype(str).str.lower().isin(["true", "1", "yes", "y"])

def smoke_sample(df, n, seed):
    if n is None or len(df) <= n:
        return df.copy().reset_index(drop=True)

    tmp = df.copy()
    tmp["_stratum"] = tmp.domain.astype(str) + "|" + tmp.decision.astype(str)
    groups = list(tmp.groupby("_stratum", sort=False))
    rng = np.random.RandomState(seed)
    per = max(1, n // max(len(groups),1))

    parts = []
    for _,g in groups:
        parts.append(
            g.sample(
                min(per, len(g)),
                random_state=int(rng.randint(2**31-1))
            )
        )

    out = pd.concat(parts).drop(columns="_stratum").drop_duplicates("id")

    if len(out) < n:
        rem = df[~df.id.isin(out.id)]
        if len(rem):
            out = pd.concat([
                out,
                rem.sample(
                    min(n-len(out), len(rem)),
                    random_state=seed
                )
            ])

    return out.sample(
        min(n, len(out)),
        random_state=seed
    ).reset_index(drop=True)

def build_lodo_split(target_domain):
    source_domains = [d for d in DOMAINS if d != target_domain]

    train = MASTER[
        (MASTER["split"] == "train") &
        (MASTER["domain"].isin(source_domains))
    ].copy()

    val = MASTER[
        (MASTER["split"] == "val") &
        (MASTER["domain"].isin(source_domains))
    ].copy()

    test = MASTER[
        (MASTER["split"] == "heldout_test") &
        (MASTER["domain"] == target_domain)
    ].copy()

    matched = test[
        bool_series(test["in_decision_matched_test"])
    ].copy()

    assert target_domain not in set(train.domain)
    assert target_domain not in set(val.domain)
    assert set(test.domain) == {target_domain}

    assert set(train.id).isdisjoint(set(val.id))
    assert set(train.id).isdisjoint(set(test.id))
    assert set(val.id).isdisjoint(set(test.id))

    assert set(train.scenario_family_id).isdisjoint(
        set(val.scenario_family_id)
    )
    assert set(train.scenario_family_id).isdisjoint(
        set(test.scenario_family_id)
    )
    assert set(val.scenario_family_id).isdisjoint(
        set(test.scenario_family_id)
    )

    for v in TEXT_VIEWS:
        tr = set(train[v].astype(str))
        assert tr.isdisjoint(
            set(val[v].astype(str))
        ), f"{target_domain}: train/val text leakage in {v}"
        assert tr.isdisjoint(
            set(test[v].astype(str))
        ), f"{target_domain}: train/test text leakage in {v}"

    if SMOKE_RUN:
        train = smoke_sample(train, SMOKE_TRAIN_N, 101)
        val = smoke_sample(val, SMOKE_VAL_N, 102)
        test = smoke_sample(test, SMOKE_TEST_N, 103)
        matched = smoke_sample(matched, SMOKE_MATCHED_N, 104)

    return {
        "target": target_domain,
        "sources": source_domains,
        "train": train.reset_index(drop=True),
        "val": val.reset_index(drop=True),
        "test": test.reset_index(drop=True),
        "matched": matched.reset_index(drop=True),
    }

LODO_SPLITS = {
    d: build_lodo_split(d)
    for d in LODO_TARGETS
}

print("\nLODO audit:")
for target,s in LODO_SPLITS.items():
    print(
        f"{target:10s} | sources={s['sources']} | "
        f"train={len(s['train'])} | val={len(s['val'])} | "
        f"test={len(s['test'])} | matched={len(s['matched'])}"
    )

print("\nLODO leakage checks: PASS")

In [ ]:
# ============================================================
# CELL 3 — SECURITY NORMALIZER + TRAIN-ONLY AUGMENTATION
# ============================================================
_ZW = re.compile("[\u200b\u2060\ufeff]")  # keep ZWNJ/ZWJ
_HOMO = str.maketrans({
    "а":"a","е":"e","о":"o","р":"p","с":"c","х":"x","і":"i","ѕ":"s",
    "ο":"o","α":"a","ε":"e","ρ":"p"
})
_ASCII_TO_HOMO = {"a":"а","e":"е","o":"о","p":"р","c":"с","x":"х","i":"і","s":"ѕ"}

def normalize_security_text(text):
    t = unicodedata.normalize("NFKC", str(text))
    t = _ZW.sub("", t).translate(_HOMO)
    frag = re.compile(r"(?i)\b(?:[a-z][\s\-._*]){3,}[a-z]\b")
    t = frag.sub(lambda m: re.sub(r"[\s\-._*]", "", m.group(0)), t)
    return re.sub(r"\s+", " ", t).strip()

BANGLISH_RULES = [
    ("sh","s"),("s","sh"),("chh","ch"),("ch","chh"),("bh","v"),("v","bh"),
    ("ph","f"),("f","ph"),("kh","k"),("gh","g"),("th","t"),("dh","d"),
    ("aa","a"),("ee","i"),("ii","i"),("oo","u"),("ou","o"),("au","o"),
    ("achhe","ache"),("ache","ase"),("ase","ache"),("nai","nei"),("nei","nai"),
    ("koro","kro"),("kro","koro"),("koru","koro"),("hobe","hbe"),("hbe","hobe"),
    ("amar","amr"),("tomar","tmr"),("apnar","apnr"),("account","acount"),
    ("password","passwrd"),("verify","verfy"),("verification","verfication"),
    ("please","pls"),("plz","pls"),("message","msg"),("number","num"),
    ("transaction","transction"),("payment","paymnt"),("transfer","transfr"),
    ("security","securty"),("system","systm"),("prompt","promt"),
    ("instruction","instrction"),("kivabe","kibhabe"),("kibhabe","kivabe"),
    ("ki","kii"),("ami","amii"),("taka","tk"),("ekhon","ekhn"),
    ("lagbe","lgbe"),("dorkar","drkar"),("korte","krte"),("korbo","krbo"),
    ("bolo","blo"),("dao","daw"),("diye","dia"),("jabe","jbe"),
    ("parbo","prbo"),("chai","chy"),
]

LOANWORD_FLIPS = {
    "debit card":"ডেবিট কার্ড","credit card":"ক্রেডিট কার্ড",
    "account":"অ্যাকাউন্ট","otp":"ওটিপি","password":"পাসওয়ার্ড",
    "refund":"রিফান্ড","invoice":"ইনভয়েস","payment":"পেমেন্ট",
    "transaction":"ট্রানজ্যাকশন","transfer":"ট্রান্সফার",
    "security":"সিকিউরিটি","system":"সিস্টেম","prompt":"প্রম্পট",
    "login":"লগইন","verification":"ভেরিফিকেশন","email":"ইমেইল",
    "sms":"এসএমএস","api":"এপিআই",
}

CARRIERS = [
    "SMS content: {x}",
    "Email body: {x}",
    "Invoice note: {x}",
    "Résumé text: {x}",
    "JSON field `message`: {x}",
    "Web snippet: {x}",
    "Vendor memo: {x}",
]

def _obfuscate(t, rng):
    toks = t.split()
    if not toks:
        return t
    mode = rng.choice([
        "fragment","spacing","zero_width","homoglyph",
        "punct","repeat","hasanta_drop"
    ])
    j = rng.randrange(len(toks))
    tok = toks[j]

    if mode == "fragment" and tok.isascii() and tok.isalpha() and len(tok) >= 5:
        toks[j] = "-".join(tok)
    elif mode == "spacing" and tok.isascii() and tok.isalpha() and len(tok) >= 5:
        toks[j] = " ".join(tok)
    elif mode == "zero_width" and len(tok) >= 4:
        k = max(1, len(tok)//2)
        toks[j] = tok[:k] + "\u200b" + tok[k:]
    elif mode == "homoglyph" and tok.isascii():
        chars = list(tok)
        cand = [i for i,c in enumerate(chars) if c.lower() in _ASCII_TO_HOMO]
        if cand:
            k = rng.choice(cand)
            chars[k] = _ASCII_TO_HOMO[chars[k].lower()]
            toks[j] = "".join(chars)
    elif mode == "punct" and len(tok) >= 4:
        toks[j] = tok + "..."
    elif mode == "repeat" and len(tok) >= 3:
        k = rng.randrange(len(tok))
        toks[j] = tok[:k] + tok[k]*2 + tok[k:]
    elif mode == "hasanta_drop" and "্" in tok:
        toks[j] = tok.replace("্", "", 1)
    return " ".join(toks)

def augment_text(text, view, intent, rng):
    original = str(text)
    t = original

    if view in ("banglish", "code_mix"):
        for _ in range(rng.randint(1,3)):
            a,b = rng.choice(BANGLISH_RULES)
            t = re.sub(re.escape(a), b, t, count=1, flags=re.I)

    if view in ("bangla", "code_mix"):
        flip_p = rng.uniform(0.20,0.50)
        for a,b in LOANWORD_FLIPS.items():
            if rng.random() < flip_p:
                t = re.sub(re.escape(a), b, t, flags=re.I)

    t = _obfuscate(t, rng)

    if intent in ("S6_indirect_injection", "B0_legitimate_security") and rng.random() < 0.50:
        t = rng.choice(CARRIERS).format(x=t)

    if t == original:
        t = original + " ."
    return t

print(normalize_security_text("s-y-s-t-e-m   prompt দেখাও"))

In [ ]:
# ============================================================
# CELL 4 — TOKENIZER + LENGTH CHECK
# ============================================================
TOKENIZER = AutoTokenizer.from_pretrained(BACKBONE, use_fast=True)

def length_audit(df):
    lens = []
    for v in TEXT_VIEWS:
        enc = TOKENIZER(
            df[v].astype(str).tolist(),
            add_special_tokens=True,
            truncation=False,
        )
        lens.extend(len(x) for x in enc["input_ids"])
    arr = np.asarray(lens)
    return {
        "p95": float(np.percentile(arr,95)),
        "p99": float(np.percentile(arr,99)),
        "max": int(arr.max()),
        "fraction_over_128": float((arr > 128).mean()),
    }

print("source-train length audit:", length_audit(MASTER[MASTER.split=="train"]))

In [ ]:
# ============================================================
# CELL 5 — SOURCE-DOMAIN-ONLY TAPT/TLM
# ============================================================
class MLMTextDataset(Dataset):
    def __init__(self, texts):
        self.texts = texts
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, i):
        return TOKENIZER(
            self.texts[i],
            truncation=True,
            max_length=MAX_LEN,
            return_special_tokens_mask=True,
        )

def build_tapt_texts(df):
    sep = TOKENIZER.sep_token or "</s>"
    texts = []
    for _,r in df.iterrows():
        views = [normalize_security_text(r[v]) for v in TEXT_VIEWS]
        texts.extend(views)
        texts.append(views[0] + f" {sep} " + views[1])  # Bangla || Banglish
        texts.append(views[1] + f" {sep} " + views[3])  # Banglish || English
        texts.append(views[2] + f" {sep} " + views[0])  # Code-mix || Bangla
    return texts

def prepare_tapt_backbone(train_df, target_domain):
    if not USE_TAPT_TLM:
        return BACKBONE

    fold_tag = target_domain.replace("-", "")
    adapted_dir = OUT_DIR / f"tapt_excluding_{fold_tag}"

    if (adapted_dir / "config.json").exists():
        print("TAPT/TLM cache:", adapted_dir)
        return str(adapted_dir)

    print(f"TAPT/TLM: source domains only; target excluded = {target_domain}")
    mlm = AutoModelForMaskedLM.from_pretrained(BACKBONE).to(DEVICE)
    if hasattr(mlm, "gradient_checkpointing_enable"):
        mlm.gradient_checkpointing_enable()

    texts = build_tapt_texts(train_df)
    if SMOKE_RUN:
        texts = texts[:128]

    ds = MLMTextDataset(texts)
    collator = DataCollatorForLanguageModeling(
        tokenizer=TOKENIZER,
        mlm_probability=0.15,
    )
    gen = torch.Generator()
    gen.manual_seed(2026)
    dl = DataLoader(
        ds,
        batch_size=TAPT_BATCH,
        shuffle=True,
        collate_fn=collator,
        generator=gen,
    )

    epochs = 1 if SMOKE_RUN else TAPT_EPOCHS
    max_updates = SMOKE_TAPT_STEPS if SMOKE_RUN else None
    updates_per_epoch = max(1, math.ceil(len(dl)/TAPT_GRAD_ACCUM))
    total_updates = max_updates if max_updates else updates_per_epoch * epochs

    opt = torch.optim.AdamW(
        mlm.parameters(),
        lr=TAPT_LR,
        weight_decay=WEIGHT_DECAY,
        foreach=False,
    )
    sched = get_linear_schedule_with_warmup(
        opt,
        int(WARMUP_RATIO * total_updates),
        total_updates,
    )
    scaler = torch.cuda.amp.GradScaler(enabled=USE_SCALER)

    opt.zero_grad(set_to_none=True)
    update = 0
    stop = False

    for ep in range(epochs):
        mlm.train()
        for bi,batch in enumerate(dl):
            batch = {k:v.to(DEVICE) for k,v in batch.items()}
            with amp_context():
                loss = mlm(**batch).loss / TAPT_GRAD_ACCUM
            scaler.scale(loss).backward()

            if (bi+1) % TAPT_GRAD_ACCUM == 0 or (bi+1) == len(dl):
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(mlm.parameters(), MAX_GRAD_NORM)
                scaler.step(opt)
                scaler.update()
                sched.step()
                opt.zero_grad(set_to_none=True)
                update += 1

                if update % 20 == 0 or (max_updates and update >= max_updates):
                    print(
                        f"  TAPT epoch={ep+1} update={update} "
                        f"loss={float(loss.detach())*TAPT_GRAD_ACCUM:.4f}"
                    )

                if max_updates and update >= max_updates:
                    stop = True
                    break
        if stop:
            break

    adapted_dir.mkdir(parents=True, exist_ok=True)
    mlm.save_pretrained(adapted_dir, safe_serialization=True)
    TOKENIZER.save_pretrained(adapted_dir)

    del mlm, opt, sched, scaler, dl, ds
    free_memory()
    return str(adapted_dir)

print("TAPT/TLM helper ready")

In [ ]:
# ============================================================
# CELL 6 — TRAIN-ONLY RATIONALE EMBEDDINGS
# ============================================================
def mean_pool(H, mask):
    m = mask.unsqueeze(-1).to(H.dtype)
    return (H*m).sum(1) / m.sum(1).clamp_min(1)

def build_note_embeddings(train_df, target_domain):
    if not USE_RATIONALE_ALIGNMENT:
        return None, 0

    fold_tag = target_domain.replace("-", "")
    cache = OUT_DIR / f"note_embeddings_excluding_{fold_tag}.npz"

    unique = train_df[["id","note"]].drop_duplicates("id").copy()
    unique["id"] = unique["id"].astype(str)
    unique["note"] = unique["note"].fillna("").astype(str)

    if cache.exists():
        x = np.load(cache, allow_pickle=True)
        ids = x["ids"].astype(str).tolist()
        emb = x["emb"].astype("float32")
        assert set(ids) == set(unique.id), "Rationale cache does not match this LODO fold"
        print("rationale cache:", cache)
        return {k:emb[i] for i,k in enumerate(ids)}, emb.shape[1]

    print(f"Encoding source-domain TRAIN notes; target excluded = {target_domain}")
    tok = AutoTokenizer.from_pretrained(RATIONALE_MODEL)
    enc = AutoModel.from_pretrained(
        RATIONALE_MODEL,
        add_pooling_layer=False,
        torch_dtype=AMP_DTYPE if torch.cuda.is_available() else torch.float32,
    ).to(DEVICE)
    enc.eval()

    notes = ["passage: " + x for x in unique.note.tolist()]
    vecs = []
    bs = 4

    for i in range(0, len(notes), bs):
        b = tok(
            notes[i:i+bs],
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt",
        )
        b = {k:v.to(DEVICE) for k,v in b.items()}
        with torch.no_grad(), amp_context():
            H = enc(**b).last_hidden_state
        z = F.normalize(mean_pool(H.float(), b["attention_mask"]), dim=-1)
        vecs.append(z.cpu().numpy())

    emb = np.concatenate(vecs, axis=0).astype("float32")
    np.savez_compressed(cache, ids=unique.id.values, emb=emb)

    out = {k:emb[i] for i,k in enumerate(unique.id.tolist())}
    dim = emb.shape[1]

    del enc, tok
    free_memory()
    return out, dim

print("rationale helper ready")

In [ ]:
# ============================================================
# CELL 7 — 4-VIEW SCENARIO DATASET
# ============================================================
class ScenarioDataset(Dataset):
    def __init__(self, df, augment=False, seed=42):
        self.df = df.reset_index(drop=True)
        self.augment = augment
        self.seed = seed

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        r = self.df.iloc[i]
        rng = random.Random(self.seed*1000003 + i)

        texts = []
        for v in TEXT_VIEWS:
            t = normalize_security_text(r[v])
            if self.augment and rng.random() < AUGMENT_PROB:
                t = augment_text(t, v, r.security_intent, rng)
            texts.append(t)

        return {
            "texts": texts,
            "id": str(r.id),
            "family": str(r.scenario_family_id),
            "decision": LABEL2ID[r.decision],
            "intent": INTENT2ID[r.security_intent],
            "domain": DOMAIN2ID[r.domain],
            "risk": RISK2ID[r.risk_level],
        }

def make_collate(training_aux=False, note_map=None):
    def collate(batch):
        texts = [t for x in batch for t in x["texts"]]
        enc = TOKENIZER(
            texts,
            padding=True,
            truncation=True,
            max_length=MAX_LEN,
            return_tensors="pt",
        )

        B = len(batch)
        for k in ["decision","intent","domain","risk"]:
            vals = torch.tensor([x[k] for x in batch], dtype=torch.long)
            enc[k] = vals.repeat_interleave(4)

        enc["scenario"] = torch.arange(B, dtype=torch.long).repeat_interleave(4)
        enc["ids"] = [x["id"] for x in batch]
        enc["families"] = [x["family"] for x in batch]

        if training_aux and USE_RATIONALE_ALIGNMENT:
            assert note_map is not None
            missing = [x["id"] for x in batch if x["id"] not in note_map]
            assert not missing, f"Missing rationale embedding: {missing[:3]}"
            ne = np.stack([note_map[x["id"]] for x in batch])
            enc["note_emb"] = torch.tensor(ne, dtype=torch.float32).repeat_interleave(4, 0)

        return enc
    return collate

# Evaluation must never require privileged training-only note embeddings.
_probe = ScenarioDataset(next(iter(LODO_SPLITS.values()))["val"].head(1))[0]
_probe_batch = make_collate(training_aux=False)([_probe])
assert "note_emb" not in _probe_batch
print("privileged-supervision isolation: PASS")

In [ ]:
# ============================================================
# CELL 8 — BANGLAGUARD XLM-R BASE + LOSSES
# ============================================================
class BanglaGuardXLMRBase(nn.Module):
    def __init__(
        self,
        backbone,
        rat_dim,
        n_dom=3,
        n_int=7,
        n_dec=3,
        n_risk=5,
        pol_dim=64,
        proj_dim=128,
        dropout=0.1,
    ):
        super().__init__()

        self.enc = AutoModel.from_pretrained(
            backbone,
            add_pooling_layer=False,
        )
        d = self.enc.config.hidden_size
        self.n_dom = n_dom

        self.attn = nn.Linear(d, 1)
        self.pool = nn.Sequential(
            nn.Linear(2*d, d),
            nn.GELU(),
            nn.LayerNorm(d),
        )
        self.drop = nn.Dropout(dropout)

        self.dom_head = nn.Linear(d, n_dom)
        self.int_head = nn.Linear(d, n_int)
        self.risk_head = nn.Linear(d, n_risk-1)  # CORN: K-1 logits

        self.policy = nn.Embedding(n_dom+1, pol_dim)  # index 3 = generic policy
        self.film = nn.Linear(pol_dim, 2*d)
        nn.init.zeros_(self.film.weight)
        nn.init.zeros_(self.film.bias)

        self.dec_head = nn.Sequential(
            nn.Linear(d+n_int, d),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d, n_dec),
        )

        self.proj = nn.Sequential(
            nn.Linear(d, d),
            nn.GELU(),
            nn.Linear(d, proj_dim),
        )
        self.rat = nn.Linear(d, rat_dim) if rat_dim > 0 else None

    def forward(
        self,
        input_ids,
        attention_mask,
        domain_id=None,
        policy_dropout=0.0,
    ):
        H = self.enc(
            input_ids=input_ids,
            attention_mask=attention_mask,
        ).last_hidden_state

        # T4/FP16-safe attention pooling.
        a = self.attn(H).squeeze(-1).float()
        a = a.masked_fill(attention_mask == 0, -1e4)
        Hf = H.float()
        att = (a.softmax(-1).unsqueeze(-1) * Hf).sum(1)
        cls = Hf[:,0]
        h = self.drop(self.pool(torch.cat([cls, att], -1)))

        dom_logits = self.dom_head(h)
        int_logits = self.int_head(h)
        risk_logits = self.risk_head(h)

        # Benchmark/source-domain path uses predicted domain.
        # LODO target inference supplies domain_id == n_dom -> generic policy.
        if domain_id is not None:
            w = F.one_hot(domain_id, self.n_dom+1).to(h.dtype)
        else:
            p_dom = dom_logits.detach().softmax(-1)
            w = torch.cat(
                [p_dom, p_dom.new_zeros(p_dom.size(0),1)],
                dim=-1,
            )

        # Policy dropout explicitly trains the generic policy embedding.
        if self.training and policy_dropout > 0:
            generic = F.one_hot(
                torch.tensor(self.n_dom, device=h.device),
                self.n_dom+1,
            ).to(h.dtype)
            mask = torch.rand(h.size(0),1,device=h.device) < policy_dropout
            w = torch.where(mask, generic.expand_as(w), w)

        gamma, beta = self.film(
            w @ self.policy.weight.to(h.dtype)
        ).chunk(2, dim=-1)

        h_pol = h*(1+gamma) + beta
        dec_logits = self.dec_head(
            torch.cat([h_pol, int_logits.softmax(-1)], dim=-1)
        )

        return {
            "h": h,
            "dec": dec_logits,
            "int": int_logits,
            "dom": dom_logits,
            "risk": risk_logits,
        }

def corn_loss(logits, y):
    losses = []
    count = 0
    for k in range(logits.size(1)):
        m = y >= k
        if m.any():
            losses.append(
                F.binary_cross_entropy_with_logits(
                    logits[m,k],
                    (y[m] > k).float(),
                    reduction="sum",
                )
            )
            count += int(m.sum())
    if not losses:
        return logits.new_tensor(0.0)
    return torch.stack(losses).sum() / max(count,1)

def multiview_js(logits, n_views=4):
    ctx = torch.autocast("cuda", enabled=False) if logits.is_cuda else nullcontext()
    with ctx:
        x = logits.float()
        logp = x.log_softmax(-1).view(-1, n_views, x.size(-1))
        p = logp.exp()
        logm = p.mean(1, keepdim=True).clamp_min(1e-8).log()
        return (p*(logp-logm)).sum(-1).mean()

def supcon(z, group, tau=0.07):
    ctx = torch.autocast("cuda", enabled=False) if z.is_cuda else nullcontext()
    with ctx:
        zf = F.normalize(z.float(), dim=-1)
        gf = group.to(zf.device)
        sim = (zf @ zf.T) / float(tau)
        eye = torch.eye(len(zf), dtype=torch.bool, device=zf.device)
        sim = sim.masked_fill(eye, -1e4)
        pos = (gf[:,None] == gf[None,:]) & ~eye
        denom = pos.sum(1)
        valid = denom > 0
        if not valid.any():
            return zf.new_tensor(0.0)
        logp = sim - sim.logsumexp(1, keepdim=True)
        return -(logp[valid]*pos[valid]).sum(1).div(denom[valid]).mean()

def total_loss(out, batch, model):
    ce = F.cross_entropy(
        out["dec"].float(),
        batch["decision"],
        weight=CLASS_W.to(out["dec"].device),
        label_smoothing=LABEL_SMOOTHING,
    )

    li = F.cross_entropy(
        out["int"].float(),
        batch["intent"],
        label_smoothing=LABEL_SMOOTHING,
    )
    ld = F.cross_entropy(out["dom"].float(), batch["domain"])
    lo = corn_loss(out["risk"].float(), batch["risk"])
    lc = multiview_js(out["dec"]) + 0.5*multiview_js(out["int"])
    ls = supcon(model.proj(out["h"]), batch["scenario"], SUPCON_TAU)

    loss = (
        ce
        + LAMBDA_INT*li
        + LAMBDA_DOM*ld
        + LAMBDA_ORD*lo
        + LAMBDA_CONS*lc
        + LAMBDA_SCL*ls
    )

    if USE_RATIONALE_ALIGNMENT:
        rat = F.cosine_similarity(
            model.rat(out["h"]).float(),
            batch["note_emb"],
            dim=-1,
        )
        loss = loss + LAMBDA_RAT*(1-rat).mean()

    return loss

print("BanglaGuard architecture ready")

In [ ]:
# ============================================================
# CELL 9 — AMP REGRESSION TEST
# ============================================================
_z = torch.randn(
    16, 128,
    device=DEVICE,
    dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
)
_g = torch.arange(4, device=DEVICE).repeat_interleave(4)
_test_loss = supcon(_z, _g, 0.07)
assert torch.isfinite(_test_loss)
print("AMP SupCon regression test: PASS | loss =", float(_test_loss))
del _z, _g, _test_loss
free_memory()

In [ ]:
# ============================================================
# CELL 10 — TRAINING UTILITIES
# ============================================================
def build_model(active_backbone, rat_dim):
    seed_all(FIXED_HEAD_INIT_SEED)
    model = BanglaGuardXLMRBase(
        active_backbone,
        rat_dim=rat_dim if USE_RATIONALE_ALIGNMENT else 0,
        dropout=DROPOUT,
    )
    if hasattr(model.enc, "gradient_checkpointing_enable"):
        model.enc.gradient_checkpointing_enable()
    return model.to(DEVICE)

def optimizer_groups(model):
    n_layers = int(getattr(model.enc.config, "num_hidden_layers", 12))
    groups = {}

    for name,p in model.named_parameters():
        if not p.requires_grad:
            continue

        wd = 0.0 if (
            "bias" in name or
            "LayerNorm" in name or
            "layer_norm" in name.lower()
        ) else WEIGHT_DECAY

        if not name.startswith("enc."):
            lr = LR_HEADS
        else:
            m = re.search(r"encoder\.layer\.(\d+)", name)
            if m:
                layer = int(m.group(1))
                lr = LR_ENCODER * (LAYERWISE_LR_DECAY ** (n_layers-1-layer))
            elif "embeddings" in name:
                lr = LR_ENCODER * (LAYERWISE_LR_DECAY ** n_layers)
            else:
                lr = LR_ENCODER

        key = (round(lr,12), wd)
        groups.setdefault(key, []).append(p)

    return [
        {"params": ps, "lr": lr, "weight_decay": wd}
        for (lr,wd), ps in groups.items()
    ]

@torch.no_grad()
def predict(model, df, generic_policy=True):
    model.eval()
    ds = ScenarioDataset(df, augment=False)
    dl = DataLoader(
        ds,
        batch_size=EVAL_SCENARIO_BATCH,
        shuffle=False,
        collate_fn=make_collate(training_aux=False),
    )

    recs = []
    offset = 0

    for batch in dl:
        ids = batch.pop("ids")
        fams = batch.pop("families")
        batch.pop("scenario")
        batch.pop("decision")
        batch.pop("intent")
        batch.pop("domain")
        batch.pop("risk")

        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)

        did = None
        if generic_policy:
            did = torch.full(
                (input_ids.shape[0],),
                len(DOMAINS),  # generic policy index = 3
                dtype=torch.long,
                device=DEVICE,
            )

        with amp_context():
            out = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                domain_id=did,
                policy_dropout=0.0,
            )

        probs = torch.softmax(out["dec"].float(), dim=-1).cpu().numpy()

        B = len(ids)
        for bi in range(B):
            src = df.iloc[offset+bi]
            for vi,view in enumerate(TEXT_VIEWS):
                j = bi*4 + vi
                pred = int(probs[j].argmax())
                recs.append({
                    "id": ids[bi],
                    "scenario_family_id": fams[bi],
                    "domain": src.domain,
                    "security_intent": src.security_intent,
                    "decision_true": src.decision,
                    "hard_negative_type": src.hard_negative_type,
                    "hard_negative": src.hard_negative_type == "hard_negative",
                    "text_view": view,
                    "y_true": LABEL2ID[src.decision],
                    "pred_id": pred,
                    "pred_decision": ID2LABEL[pred],
                    "p_ALLOW": float(probs[j,0]),
                    "p_REVIEW": float(probs[j,1]),
                    "p_BLOCK": float(probs[j,2]),
                })
        offset += B

    assert len(recs) == len(df)*4
    return pd.DataFrame(recs)

def source_val_macro_f1(model, val_df):
    # Source-domain validation uses normal predicted-domain policy path.
    model.eval()
    ds = ScenarioDataset(val_df, augment=False)
    dl = DataLoader(
        ds,
        batch_size=EVAL_SCENARIO_BATCH,
        shuffle=False,
        collate_fn=make_collate(training_aux=False),
    )
    ys, ps = [], []

    with torch.no_grad():
        for batch in dl:
            y = batch["decision"].numpy()
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            with amp_context():
                out = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    domain_id=None,
                    policy_dropout=0.0,
                )
            pred = out["dec"].float().argmax(-1).cpu().numpy()
            ys.extend(y.tolist())
            ps.extend(pred.tolist())

    return float(
        f1_score(
            ys, ps,
            average="macro",
            labels=[0,1,2],
            zero_division=0,
        )
    )

def train_seed(
    train_df,
    val_df,
    active_backbone,
    note_map,
    rat_dim,
    seed,
    target_domain,
):
    model = build_model(active_backbone, rat_dim)
    seed_all(seed)

    ds = ScenarioDataset(train_df, augment=True, seed=seed)
    gen = torch.Generator()
    gen.manual_seed(seed)
    dl = DataLoader(
        ds,
        batch_size=TRAIN_SCENARIO_BATCH,
        shuffle=True,
        generator=gen,
        collate_fn=make_collate(
            training_aux=True,
            note_map=note_map,
        ),
    )

    epochs = 1 if SMOKE_RUN else EPOCHS
    max_updates = SMOKE_MAX_STEPS if SMOKE_RUN else None
    updates_per_epoch = max(1, math.ceil(len(dl)/GRAD_ACCUM))
    total_updates = max_updates if max_updates else updates_per_epoch*epochs

    opt = torch.optim.AdamW(
        optimizer_groups(model),
        foreach=False,
    )
    sched = get_linear_schedule_with_warmup(
        opt,
        int(WARMUP_RATIO*total_updates),
        total_updates,
    )
    scaler = torch.cuda.amp.GradScaler(enabled=USE_SCALER)

    best_f1 = -1.0
    best_state = None
    best_epoch = -1
    stale = 0
    update = 0
    history = []

    for ep in range(epochs):
        model.train()
        opt.zero_grad(set_to_none=True)
        losses = []
        stop = False

        for bi,batch in enumerate(dl):
            batch.pop("ids")
            batch.pop("families")
            batch = {
                k:(v.to(DEVICE) if torch.is_tensor(v) else v)
                for k,v in batch.items()
            }

            with amp_context():
                out = model(
                    input_ids=batch["input_ids"],
                    attention_mask=batch["attention_mask"],
                    domain_id=None,
                    policy_dropout=POLICY_DROPOUT,
                )
                loss = total_loss(out, batch, model) / GRAD_ACCUM

            scaler.scale(loss).backward()
            losses.append(float(loss.detach())*GRAD_ACCUM)

            if (bi+1)%GRAD_ACCUM == 0 or (bi+1) == len(dl):
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
                scaler.step(opt)
                scaler.update()
                sched.step()
                opt.zero_grad(set_to_none=True)
                update += 1

                if max_updates and update >= max_updates:
                    stop = True
                    break

        vf1 = source_val_macro_f1(model, val_df)
        row = {
            "target_domain": target_domain,
            "seed": seed,
            "epoch": ep+1,
            "updates": update,
            "train_loss": float(np.mean(losses)),
            "source_val_macro_f1": vf1,
        }
        history.append(row)
        print(
            f"    epoch={ep+1:02d} updates={update:04d} "
            f"loss={row['train_loss']:.4f} source-val-F1={vf1:.4f}"
        )

        if vf1 > best_f1 + EARLY_STOP_MIN_DELTA:
            best_f1 = vf1
            best_epoch = ep+1
            stale = 0
            best_state = {
                k:v.detach().cpu().clone()
                for k,v in model.state_dict().items()
            }
        else:
            if vf1 > best_f1:
                best_f1 = vf1
                best_epoch = ep+1
                best_state = {
                    k:v.detach().cpu().clone()
                    for k,v in model.state_dict().items()
                }
            stale += 1

        if stop:
            break
        if not SMOKE_RUN and stale >= PATIENCE:
            print(
                f"    early stop: best epoch={best_epoch}, "
                f"source-val macro-F1={best_f1:.4f}"
            )
            break

    assert best_state is not None
    model.load_state_dict(best_state, strict=True)

    # Release Adam moment tensors and the duplicate CPU checkpoint before
    # target-domain evaluation / the next seed.
    del best_state, opt, sched, scaler, dl, ds
    free_memory()

    return model, pd.DataFrame(history), best_f1

print("training utilities ready")

In [ ]:
# ============================================================
# CELL 11 — CROSS-DOMAIN METRICS
# ============================================================
def metrics_from_predictions(df):
    y = df.y_true.values.astype(int)
    p = df.pred_id.values.astype(int)

    pr, rc, f1, sup = precision_recall_fscore_support(
        y, p,
        labels=[0,1,2],
        zero_division=0,
    )

    yb = (y != LABEL2ID["ALLOW"]).astype(int)
    pb = (p != LABEL2ID["ALLOW"]).astype(int)

    out = {
        "n_prompts": int(len(df)),
        "macro_f1": float(
            f1_score(y,p,average="macro",labels=[0,1,2],zero_division=0)
        ),
        "accuracy": float(accuracy_score(y,p)),
        "balanced_accuracy": float(balanced_accuracy_score(y,p)),
        "mcc": float(matthews_corrcoef(y,p)),
        "f1_ALLOW": float(f1[0]),
        "f1_REVIEW": float(f1[1]),
        "f1_BLOCK": float(f1[2]),
        "recall_ALLOW": float(rc[0]),
        "recall_REVIEW": float(rc[1]),
        "recall_BLOCK": float(rc[2]),
        "binary_macro_f1": float(
            f1_score(yb,pb,average="macro",zero_division=0)
        ),
        "threat_recall": float(
            ((pb==1)&(yb==1)).sum() / max((yb==1).sum(),1)
        ),
    }

    hard = df.hard_negative.values.astype(bool)
    out["over_refusal"] = (
        float((p[hard] != LABEL2ID["ALLOW"]).mean())
        if hard.any() else np.nan
    )
    return out

def per_view_metrics(df):
    rows = []
    for view,g in df.groupby("text_view"):
        m = metrics_from_predictions(g)
        rows.append({
            "text_view": view,
            "macro_f1": m["macro_f1"],
            "f1_REVIEW": m["f1_REVIEW"],
            "over_refusal": m["over_refusal"],
            "n_prompts": len(g),
        })
    return pd.DataFrame(rows)

def family_bootstrap_macro_f1(df, n_boot=N_BOOT, seed=42):
    fams = np.asarray(df.scenario_family_id.unique(), dtype=object)
    fam_to_idx = {
        f:np.where(df.scenario_family_id.values == f)[0]
        for f in fams
    }
    y = df.y_true.values.astype(int)
    p = df.pred_id.values.astype(int)

    rng = np.random.default_rng(seed)
    vals = []
    for _ in range(n_boot):
        sampled = rng.choice(fams, size=len(fams), replace=True)
        idx = np.concatenate([fam_to_idx[f] for f in sampled])
        vals.append(
            f1_score(
                y[idx], p[idx],
                average="macro",
                labels=[0,1,2],
                zero_division=0,
            )
        )
    return [
        float(x)
        for x in np.percentile(vals, [2.5,50,97.5])
    ]

print("metric utilities ready")

In [ ]:
# ============================================================
# CELL 12 — RUN THE 3 LODO EXPERIMENTS
# ============================================================
summary_rows = []
all_view_rows = []
all_histories = []

for target_domain in LODO_TARGETS:
    print("\n" + "="*72)
    print("LODO TARGET:", target_domain)
    print("="*72)

    fold = LODO_SPLITS[target_domain]
    print("source domains:", fold["sources"])
    print(
        "train/val/test/matched:",
        len(fold["train"]),
        len(fold["val"]),
        len(fold["test"]),
        len(fold["matched"]),
    )

    # IMPORTANT: both resources are prepared from SOURCE TRAIN ONLY.
    active_backbone = prepare_tapt_backbone(
        fold["train"],
        target_domain,
    )
    note_map, rat_dim = build_note_embeddings(
        fold["train"],
        target_domain,
    )

    fold_dir = OUT_DIR / f"lodo_{target_domain.replace('-','')}"
    fold_dir.mkdir(parents=True, exist_ok=True)

    for seed in ACTIVE_SEEDS:
        print(f"\n  --- seed {seed} ---")
        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()

        t0 = time.time()
        free_memory()
        model, hist, best_val = train_seed(
            fold["train"],
            fold["val"],
            active_backbone,
            note_map,
            rat_dim,
            seed,
            target_domain,
        )
        train_min = (time.time()-t0)/60.0

        hist.to_csv(
            fold_dir / f"history_seed{seed}.csv",
            index=False,
        )
        all_histories.append(hist)

        # CRITICAL LODO behavior:
        # unseen target domain always uses the GENERIC policy embedding.
        test_pred = predict(
            model,
            fold["test"],
            generic_policy=True,
        )
        test_pred.to_csv(
            fold_dir / f"target_predictions_seed{seed}.csv",
            index=False,
        )

        test_m = metrics_from_predictions(test_pred)
        ci = family_bootstrap_macro_f1(
            test_pred,
            N_BOOT,
            seed,
        )

        if len(fold["matched"]):
            matched_pred = predict(
                model,
                fold["matched"],
                generic_policy=True,
            )
            matched_pred.to_csv(
                fold_dir / f"matched_predictions_seed{seed}.csv",
                index=False,
            )
            matched_m = metrics_from_predictions(matched_pred)
            matched_f1 = matched_m["macro_f1"]
        else:
            matched_f1 = np.nan

        views = per_view_metrics(test_pred)
        views["target_domain"] = target_domain
        views["seed"] = seed
        all_view_rows.append(views)

        peak_gb = (
            float(torch.cuda.max_memory_allocated()/1024**3)
            if torch.cuda.is_available() else 0.0
        )

        row = {
            "target_domain": target_domain,
            "source_domains": " + ".join(fold["sources"]),
            "seed": seed,
            "n_target_scenarios": len(fold["test"]),
            "n_target_prompts": len(test_pred),
            "best_source_val_macro_f1": best_val,
            "target_macro_f1": test_m["macro_f1"],
            "target_accuracy": test_m["accuracy"],
            "target_f1_ALLOW": test_m["f1_ALLOW"],
            "target_f1_REVIEW": test_m["f1_REVIEW"],
            "target_f1_BLOCK": test_m["f1_BLOCK"],
            "target_binary_macro_f1": test_m["binary_macro_f1"],
            "target_threat_recall": test_m["threat_recall"],
            "target_over_refusal": test_m["over_refusal"],
            "decision_matched_macro_f1": matched_f1,
            "bootstrap_f1_lo": ci[0],
            "bootstrap_f1_median": ci[1],
            "bootstrap_f1_hi": ci[2],
            "train_minutes": train_min,
            "peak_gpu_mem_gb": peak_gb,
            "generic_policy_at_test": True,
        }
        summary_rows.append(row)

        print(
            f"  TARGET {target_domain}: "
            f"macro-F1={row['target_macro_f1']:.4f} | "
            f"REVIEW-F1={row['target_f1_REVIEW']:.4f} | "
            f"matched-F1={row['decision_matched_macro_f1']:.4f} | "
            f"over-refusal={row['target_over_refusal']:.4f}"
        )

        del model, test_pred, test_m, views, hist
        if "matched_pred" in locals():
            del matched_pred
        if "matched_m" in locals():
            del matched_m
        free_memory()

    del note_map
    free_memory()

SUMMARY = pd.DataFrame(summary_rows)
VIEWS = pd.concat(all_view_rows, ignore_index=True)
HISTORY = pd.concat(all_histories, ignore_index=True)

SUMMARY.to_csv(OUT_DIR / "cross_domain_summary_by_seed.csv", index=False)
VIEWS.to_csv(OUT_DIR / "cross_domain_per_view.csv", index=False)
HISTORY.to_csv(OUT_DIR / "cross_domain_training_history.csv", index=False)

display(SUMMARY)

In [ ]:
# ============================================================
# CELL 13 — AGGREGATED CROSS-DOMAIN TABLE + FINAL PACKAGE
# ============================================================
agg = (
    SUMMARY.groupby("target_domain")
    .agg(
        macro_f1_mean=("target_macro_f1","mean"),
        macro_f1_std=("target_macro_f1","std"),
        review_f1_mean=("target_f1_REVIEW","mean"),
        review_f1_std=("target_f1_REVIEW","std"),
        matched_f1_mean=("decision_matched_macro_f1","mean"),
        matched_f1_std=("decision_matched_macro_f1","std"),
        over_refusal_mean=("target_over_refusal","mean"),
        threat_recall_mean=("target_threat_recall","mean"),
        n_seeds=("seed","nunique"),
    )
    .reset_index()
)

# Macro-average over the three unseen target domains.
cross_domain_macro = float(
    agg["macro_f1_mean"].mean()
)

agg.to_csv(
    OUT_DIR / "cross_domain_aggregated.csv",
    index=False,
)

print("\nCross-domain aggregated results:")
display(agg)
print(f"LODO macro-average across target domains: {cross_domain_macro:.4f}")

# Protocol audit.
audit = {
    "model": "BanglaGuard XLM-R Base",
    "analysis": "leave-one-domain-out cross-domain only",
    "targets": LODO_TARGETS,
    "seeds": ACTIVE_SEEDS,
    "smoke_run": SMOKE_RUN,
    "target_excluded_from_train": True,
    "target_excluded_from_validation": True,
    "target_excluded_from_tapt_tlm": True,
    "target_excluded_from_rationale_alignment": True,
    "generic_policy_embedding_used_on_target": True,
    "cross_domain_macro_average": cross_domain_macro,
}
with open(OUT_DIR / "cross_domain_protocol.json", "w") as f:
    json.dump(audit, f, indent=2)

# Small result package only; model/TAPT checkpoints remain in OUT_DIR.
report_dir = OUT_DIR / "report_files"
report_dir.mkdir(parents=True, exist_ok=True)

for p in OUT_DIR.rglob("*"):
    if p.is_file() and p.suffix.lower() in {".csv",".json"}:
        if report_dir not in p.parents:
            rel = p.relative_to(OUT_DIR)
            dst = report_dir / rel
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(p, dst)

zip_out = WORK_ROOT / f"{MODEL_SLUG}_{RUN_MODE}_cross_domain_results.zip"
shutil.make_archive(
    str(zip_out).replace(".zip",""),
    "zip",
    report_dir,
)

print("\n============================================================")
print("BANGLAGUARD XLM-R BASE CROSS-DOMAIN", "SMOKE PASSED" if SMOKE_RUN else "COMPLETE")
print("results :", OUT_DIR)
print("zip     :", zip_out)
print("============================================================")

print("\nStarting browser download...")
colab_files.download(str(zip_out))
